# 大模型与小模型结合使用策略

本文档介绍如何将大型模型（Qwen3.5-35B-A3B）和小型模型（Qwen0.8B）有效结合使用，包括多种架构设计和代码实现。

## 模型概述

### Qwen3.5-35B-A3B
- 总参数：35B
- 活跃参数：3B（MoE架构）
- 适用场景：复杂推理、多任务处理、高质量生成

### Qwen0.8B
- 总参数：0.8B
- 适用场景：简单任务、快速响应、低成本推理

---

## 1. 级联路由 (Cascade Routing)

使用小模型快速处理简单任务，遇到困难时升级到大模型。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

class ModelRouter:
    def __init__(self, small_model_path, large_model_path, small_tokenizer, large_tokenizer):
        self.small_model = AutoModelForCausalLM.from_pretrained(small_model_path, device_map="auto")
        self.large_model = AutoModelForCausalLM.from_pretrained(large_model_path, device_map="auto")
        self.small_tokenizer = small_tokenizer
        self.large_tokenizer = large_tokenizer
        self.confidence_threshold = 0.8
    
    def get_confidence(self, model, tokenizer, inputs):
        """计算模型输出的置信度"""
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.softmax(logits[:, -1, :], dim=-1)
            max_prob, _ = torch.max(probs, dim=-1)
            return max_prob.item()
    
    def generate(self, prompt, max_new_tokens=256):
        """级联生成：先尝试小模型，置信度不足则使用大模型"""
        # 小模型尝试
        inputs = self.small_tokenizer(prompt, return_tensors="pt").to(self.small_model.device)
        confidence = self.get_confidence(self.small_model, self.small_tokenizer, inputs)
        
        if confidence >= self.confidence_threshold:
            print(f"✓ 使用小模型 (置信度: {confidence:.3f})")
            outputs = self.small_model.generate(**inputs, max_new_tokens=max_new_tokens)
            return self.small_tokenizer.decode(outputs[0], skip_special_tokens=True)
        else:
            print(f"→ 升级到大型模型 (置信度: {confidence:.3f} < {self.confidence_threshold})")
            inputs = self.large_tokenizer(prompt, return_tensors="pt").to(self.large_model.device)
            outputs = self.large_model.generate(**inputs, max_new_tokens=max_new_tokens)
            return self.large_tokenizer.decode(outputs[0], skip_special_tokens=True)

# 使用示例
# router = ModelRouter(
#     small_model_path="Qwen/Qwen0.8B",
#     large_model_path="Qwen/Qwen3.5-35B-A3B",
#     small_tokenizer=AutoTokenizer.from_pretrained("Qwen/Qwen0.8B"),
#     large_tokenizer=AutoTokenizer.from_pretrained("Qwen/Qwen3.5-35B-A3B")
# )
# result = router.generate("解释量子纠缠的原理")

## 2. 任务分类路由 (Task Classification)

根据任务类型智能选择合适的模型。

In [ ]:
from typing import Dict, List

class TaskClassifier:
    def __init__(self):
        # 定义任务类型及其关键词
        self.task_patterns = {
            "simple": {
                "keywords": ["翻译", "摘要", "分类", "提取", "简单", "快速"],
                "model": "small"
            },
            "complex_reasoning": {
                "keywords": ["推理", "证明", "推导", "分析", "原理", "复杂"],
                "model": "large"
            },
            "creative": {
                "keywords": ["创作", "写诗", "故事", "小说", "创意"],
                "model": "large"
            },
            "coding": {
                "keywords": ["代码", "编程", "算法", "bug", "调试"],
                "model": "large"
            },
            "qa": {
                "keywords": ["回答", "问题", "解释", "说明"],
                "model": "small"
            }
        }
    
    def classify(self, prompt: str) -> str:
        """根据提示词分类任务类型"""
        scores = {task: 0 for task in self.task_patterns}
        
        for task, config in self.task_patterns.items():
            for keyword in config["keywords"]:
                if keyword in prompt:
                    scores[task] += 1
        
        # 返回得分最高的任务类型对应的模型
        best_task = max(scores, key=scores.get)
        return self.task_patterns[best_task]["model"]

class SmartRouter:
    def __init__(self, small_model, large_model, small_tokenizer, large_tokenizer):
        self.small_model = small_model
        self.large_model = large_model
        self.small_tokenizer = small_tokenizer
        self.large_tokenizer = large_tokenizer
        self.classifier = TaskClassifier()
    
    def generate(self, prompt: str, max_new_tokens=256):
        """根据任务类型智能路由"""
        model_type = self.classifier.classify(prompt)
        
        if model_type == "small":
            print(f"📦 使用小型模型 (任务类型: 简单任务)")
            inputs = self.small_tokenizer(prompt, return_tensors="pt").to(self.small_model.device)
            outputs = self.small_model.generate(**inputs, max_new_tokens=max_new_tokens)
            return self.small_tokenizer.decode(outputs[0], skip_special_tokens=True)
        else:
            print(f"🎯 使用大型模型 (任务类型: 复杂任务)")
            inputs = self.large_tokenizer(prompt, return_tensors="pt").to(self.large_model.device)
            outputs = self.large_model.generate(**inputs, max_new_tokens=max_new_tokens)
            return self.large_tokenizer.decode(outputs[0], skip_special_tokens=True)

## 3. 推测解码 (Speculative Decoding)

用小模型快速生成候选token，大模型验证并修正。

In [ ]:
class SpeculativeDecoder:
    def __init__(self, small_model, large_model, small_tokenizer, large_tokenizer):
        self.small_model = small_model
        self.large_model = large_model
        self.small_tokenizer = small_tokenizer
        self.large_tokenizer = large_tokenizer
        self.speculation_steps = 4  # 小模型预先生成多少个token
    
    def verify_tokens(self, prompt_tokens, candidate_tokens):
        """大模型验证小模型生成的token"""
        # 合并提示词和候选token
        full_tokens = torch.cat([prompt_tokens, candidate_tokens], dim=1)
        
        with torch.no_grad():
            outputs = self.large_model(full_tokens)
            logits = outputs.logits
            
        # 逐个验证
        accepted = []
        for i in range(candidate_tokens.shape[1]):
            # 查看大模型对这个位置的预测
            prompt_len = prompt_tokens.shape[1] + i
            pred_logits = logits[:, prompt_len - 1, :]
            pred_token = torch.argmax(pred_logits, dim=-1)
            
            # 检查小模型的token是否被接受
            if pred_token.item() == candidate_tokens[:, i].item():
                accepted.append(candidate_tokens[:, i])
            else:
                # 不接受，使用大模型的预测
                accepted.append(pred_token)
                break  # 遇到不接受就停止
        
        return torch.stack(accepted, dim=1)
    
    def generate(self, prompt: str, max_new_tokens=256):
        """使用推测解码加速生成"""
        # 编码提示词
        inputs = self.small_tokenizer(prompt, return_tensors="pt")
        prompt_tokens = inputs["input_ids"].to(self.small_model.device)
        
        generated = prompt_tokens.clone()
        total_generated = 0
        
        while total_generated < max_new_tokens:
            # 小模型快速生成候选token
            with torch.no_grad():
                small_outputs = self.small_model(generated)
                small_logits = small_outputs.logits
            
            # 采样多个候选token
            candidate_tokens = []
            current_generated = generated
            for _ in range(min(self.speculation_steps, max_new_tokens - total_generated)):
                next_token_logits = small_logits[:, -1, :]
                next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
                candidate_tokens.append(next_token)
                current_generated = torch.cat([current_generated, next_token], dim=1)
                
                # 继续预测下一个
                with torch.no_grad():
                    small_outputs = self.small_model(current_generated)
                    small_logits = small_outputs.logits
            
            candidate_tokens = torch.cat(candidate_tokens, dim=1)
            
            # 大模型验证
            accepted_tokens = self.verify_tokens(generated, candidate_tokens)
            
            # 添加接受的token
            generated = torch.cat([generated, accepted_tokens], dim=1)
            total_generated += accepted_tokens.shape[1]
        
        return self.small_tokenizer.decode(generated[0], skip_special_tokens=True)

## 4. MoE风格协作 (Mixture of Experts)

根据输入动态选择模型作为专家。

In [ ]:
class MixtureOfModels:
    def __init__(self, small_model, large_model, small_tokenizer, large_tokenizer):
        self.small_model = small_model
        self.large_model = large_model
        self.small_tokenizer = small_tokenizer
        self.large_tokenizer = large_tokenizer
        
        # 定义不同领域的专家选择规则
        self.expert_rules = {
            "math": "large",
            "science": "large",
            "code": "large",
            "translation": "small",
            "summarization": "small",
            "general_qa": "small"
        }
    
    def select_expert(self, prompt: str) -> str:
        """根据内容选择合适的专家模型"""
        prompt_lower = prompt.lower()
        
        # 数学相关
        if any(kw in prompt_lower for kw in ["数学", "计算", "方程", "公式", "证明"]):
            return self.expert_rules["math"]
        
        # 科学相关
        if any(kw in prompt_lower for kw in ["物理", "化学", "生物", "科学", "原理"]):
            return self.expert_rules["science"]
        
        # 代码相关
        if any(kw in prompt_lower for kw in ["代码", "编程", "函数", "算法", "python"]):
            return self.expert_rules["code"]
        
        # 翻译
        if "翻译" in prompt_lower or "translate" in prompt_lower:
            return self.expert_rules["translation"]
        
        # 摘要
        if "摘要" in prompt_lower or "summarize" in prompt_lower:
            return self.expert_rules["summarization"]
        
        # 默认使用小模型
        return self.expert_rules["general_qa"]
    
    def generate(self, prompt: str, max_new_tokens=256):
        """根据内容动态选择专家模型生成"""
        expert = self.select_expert(prompt)
        
        if expert == "small":
            print(f"🔹 选择小型模型专家")
            model = self.small_model
            tokenizer = self.small_tokenizer
        else:
            print(f"🔸 选择大型模型专家")
            model = self.large_model
            tokenizer = self.large_tokenizer
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)

## 5. 成本感知路由 (Cost-Aware Routing)

根据预算和优先级动态分配模型资源。

In [ ]:
from enum import Enum
from dataclasses import dataclass

class Priority(Enum):
    LOW = 1
    MEDIUM = 2
    HIGH = 3
    CRITICAL = 4

@dataclass
class Request:
    prompt: str
    priority: Priority
    user_id: str

class CostAwareRouter:
    def __init__(
        self,
        small_model, large_model,
        small_tokenizer, large_tokenizer,
        budget_limit: float = 100.0  # 预算限制（虚拟货币）
    ):
        self.small_model = small_model
        self.large_model = large_model
        self.small_tokenizer = small_tokenizer
        self.large_tokenizer = large_tokenizer
        self.budget_limit = budget_limit
        self.current_budget = budget_limit
        
        # 成本定义（每次调用的成本）
        self.costs = {
            "small": 0.1,
            "large": 1.0
        }
    
    def route(self, request: Request) -> str:
        """根据优先级和预算路由请求"""
        # 高优先级直接使用大模型
        if request.priority >= Priority.HIGH:
            if self.current_budget >= self.costs["large"]:
                return "large"
            else:
                print(f"⚠️ 预算不足，降级到小模型")
                return "small"
        
        # 中等优先级：预算充足时用大模型，否则小模型
        if request.priority == Priority.MEDIUM:
            if self.current_budget >= self.costs["large"] * 2:  # 保留更多预算
                return "large"
            return "small"
        
        # 低优先级总是用小模型
        return "small"
    
    def generate(self, request: Request, max_new_tokens=256):
        """根据路由策略生成"""
        model_type = self.route(request)
        
        # 扣除成本
        self.current_budget -= self.costs[model_type]
        
        if model_type == "small":
            print(f"💰 使用小模型 (剩余预算: {self.current_budget:.2f})")
            model = self.small_model
            tokenizer = self.small_tokenizer
        else:
            print(f"💎 使用大模型 (剩余预算: {self.current_budget:.2f})")
            model = self.large_model
            tokenizer = self.large_tokenizer
        
        inputs = tokenizer(request.prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    def reset_budget(self):
        """重置预算"""
        self.current_budget = self.budget_limit

## 综合比较

| 策略 | 优势 | 适用场景 | 实现难度 |
|------|------|----------|----------|
| 级联路由 | 自动升级，无需人工干预 | 通用场景，不确定任务难度 | 中等 |
| 任务分类路由 | 精准匹配任务类型 | 任务类型明确的场景 | 简单 |
| 推测解码 | 加速推理，提高效率 | 长文本生成 | 较难 |
| MoE协作 | 专业化处理，质量高 | 领域明确的专业场景 | 中等 |
| 成本感知 | 预算可控，资源优化 | 有预算限制的生产环境 | 中等 |

---

## 使用建议

1. **开发测试阶段**：使用任务分类路由，快速验证效果
2. **生产环境**：使用成本感知路由，平衡性能和成本
3. **追求速度**：使用推测解码，显著提升推理速度
4. **追求质量**：使用MoE协作，针对领域优化
5. **通用场景**：使用级联路由，自适应任务难度